In [1]:
"""
Simple runnable test for NBA Moneyline Betting Operator
"""
from decimal import Decimal
from agentx.samples.nba_moneyline_op import (
    BrokerOperatorConfig,
    BrokerOperator,
    BetRequest,
    StreamEvent,
)
from typing import Any
from datetime import datetime


# Create a mock Agent class for testing
class MockAgent:
    """Mock Agent implementation for testing"""

    def __init__(self, actor_id: str):
        self.actor_id = actor_id

    async def handle_stream_event(self, event: StreamEvent[Any]) -> None:
        """Process asynchronous data delivered by a DataStream."""
        winner = event.payload.get("winner")
        event_id = event.payload.get("event_id")
        print(
            f"[AGENT {self.actor_id}] Received event for {event_id}: Winner = {winner}"
        )


def create_test_broker_config(
    broker_id: str, initial_balance: str = "1000.00"
) -> BrokerOperatorConfig:
    """
    Create a simple broker configuration for testing.
    """
    return {"actor_id": broker_id, "initial_balance": initial_balance}


def create_test_broker(
    broker_id: str = "test_broker",
    initial_balance: str = "1000.00",
) -> BrokerOperator:
    """
    Create a broker for testing.
    """
    config = create_test_broker_config(broker_id, initial_balance)
    return BrokerOperator.from_dict(dict(config))


async def main():
    # Create broker
    print("1. Creating broker...")
    broker = create_test_broker()
    print(f"Broker created: {broker.actor_id}")
    print()

    # Create and register agent
    print("2. Creating and registering test agent...")
    test_agent = MockAgent(actor_id="test_agent")
    broker.register_agents([test_agent])
    balance = await broker.get_balance("test_agent")
    print("Agent registered: test_agent")
    print(f"Initial balance: ${balance}")
    print()

    # Place a bet
    print("3. Placing bet...")
    bet = await broker.place_bet(
        "test_agent",
        BetRequest(
            amount=Decimal("100.00"),
            selection="home",
            odds=Decimal("1.95"),
            event_id="lakers_vs_warriors",
        ),
    )
    print(f"Bet placed: {bet.bet_id}")
    print(f"Amount: ${bet.amount}")
    print(f"Selection: {bet.selection}")
    print(f"Odds: {bet.odds}")
    print(f"Event: {bet.event_id}")
    print()

    # Check balance after bet
    print("4. Checking balance after bet placement...")
    balance = await broker.get_balance("test_agent")
    print(f"Balance after bet: ${balance}")
    print(f"Funds locked: ${Decimal('100.00')}")
    print()

    # Check active bets
    print("5. Checking active bets...")
    active_bets = await broker.get_active_bets("test_agent")
    print(f"Active bets: {len(active_bets)}")
    print()

    # Settle the bet (home team wins)
    print("6. Settling bet (home wins)...")
    event = StreamEvent(
        stream_id="nba_results_stream",
        payload={
            "event_id": "lakers_vs_warriors",
            "winner": "home",
            "final_data": {"home_score": 110, "away_score": 105},
        },
        emitted_at=datetime.now(),
    )
    settled_count = await broker.settle_event(event)
    print(f"Bets settled: {settled_count}")
    print(f"Bet outcome: {bet.outcome.value}")
    print(f"Payout: ${bet.actual_payout}")
    print()

    # Check final balance
    print("7. Checking final balance...")
    balance = await broker.get_balance("test_agent")
    print(f"Final balance: ${balance}")
    print()

    # Get statistics
    print("8. Getting agent statistics...")
    stats = await broker.get_statistics("test_agent")
    print(f"Total bets: {stats.total_bets}")
    print(f"Total wagered: ${stats.total_wagered}")
    print(f"Wins: {stats.wins}")
    print(f"Losses: {stats.losses}")
    print(f"Win rate: {stats.win_rate:.2%}")
    print(f"Net profit: ${stats.net_profit}")
    print(f"ROI: {stats.roi:.2%}")
    print()

    # Check bet history
    print("9. Checking bet history...")
    history = await broker.get_bet_history("test_agent")
    print(f"Historical bets: {len(history)}")
    print()

    print("=" * 60)
    print("TEST COMPLETED SUCCESSFULLY")
    print("=" * 60)


await main()

1. Creating broker...
Broker created: test_broker

2. Creating and registering test agent...
[TRANSACTION] 2025-12-04T23:47:09.997643 - Agent: test_agent, Type: ACCOUNT_CREATED, Amount: 1000.00
Agent registered: test_agent
Initial balance: $1000.00

3. Placing bet...
[BET] 2025-12-04T23:47:09.997826 - Agent: test_agent, Action: BET_PLACED, BetID: f7c69616-7428-4f1e-bb4e-ba0fa159cb65, Amount: 100.00, Selection: home, Odds: 1.95
Bet placed: f7c69616-7428-4f1e-bb4e-ba0fa159cb65
Amount: $100.00
Selection: home
Odds: 1.95
Event: lakers_vs_warriors

4. Checking balance after bet placement...
Balance after bet: $900.00
Funds locked: $100.00

5. Checking active bets...
Active bets: 1

6. Settling bet (home wins)...
[SETTLEMENT] 2025-12-04T23:47:09.997877 - BetID: f7c69616-7428-4f1e-bb4e-ba0fa159cb65, EventID: lakers_vs_warriors, Winner: home, Outcome: WIN, Payout: 195.0000
[NOTIFICATION] Settling Bet f7c69616-7428-4f1e-bb4e-ba0fa159cb65 for Agent test_agent - Event lakers_vs_warriors (Winner: 